In [ ]:
from google.colab import userdata
import os
userdata.get('OPENAI')
OPENAI_API_KEY = userdata.get("OPENAI")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
!pip install langchain-chroma langchain-huggingface sentence-transformers

# 임베딩 벡터 사용 쿡북

import os
import zipfile
import torch
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings #임베딩 모델

# 1. 설정: 파일 경로 및 모델
ZIP_FILE_PATH = "/content/chroma_db_bge_m3-20260114T091037Z-3-001.zip"  # 가지고 있는 zip 파일명으로 수정 필요
EXTRACT_PATH = "./chroma_db_bge_m3_data"  # 압축 풀릴 폴더명

# 2. Zip 압축 해제

if not os.path.exists(EXTRACT_PATH):
    print(f"[{ZIP_FILE_PATH}] 압축 해제 중...")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("압축 해제 완료.")
else:
    print(f"이미 {EXTRACT_PATH} 폴더가 존재하여 압축 해제를 생략합니다.")

# 3. 임베딩 모델 로드 (BGE-M3)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"임베딩 모델 로드 중 (Device: {device})...")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True} # 코사인 유사도 검색을 위한 필수 설정
)

# 4. ChromaDB 연결
# 압축 푼 폴더 내부 구조 확인 (폴더 안에 또 폴더가 있는지 체크)
target_dir = EXTRACT_PATH
for root, dirs, files in os.walk(EXTRACT_PATH):
    if "chroma.sqlite3" in files:
        target_dir = root
        break

print(f"DB 로드 경로: {target_dir}")

vector_db = Chroma(
    persist_directory=target_dir,
    embedding_function=embedding_model,
    collection_name="multihop_rag" # 제작자가 설정한 컬렉션 이름 (이전 코드 기준)
)

# # 5. 검색 실행 (요청하신 쿼리)

# query = "Which individual is implicated in both inflating the value of a Manhattan apartment to a figure not yet achieved in New York City's real estate history, according to 'Fortune', and is also accused of adjusting this apartment's valuation to compensate for a loss in another asset's worth, as reported by 'The Age'?"

# print("\n" + "="*60)
# print(f"Q: {query}")
# print("="*60)

# # 유사도 검색 수행
# results = vector_db.similarity_search(query, k=3)

# # # 6. 결과 확인

# if not results:
#     print("검색 결과가 없습니다. (컬렉션 이름이나 임베딩 모델 설정 불일치 가능성)")

# for i, res in enumerate(results):
#     print(f"\n[Rank {i+1}]")
#     print(f"Metadata: {res.metadata}")
#     print("-" * 30)
#     print(f"Content : {res.page_content[:300]}...") # 내용이 길면 300자까지만 출력
#     print("-" * 30)




이미 ./chroma_db_bge_m3_data 폴더가 존재하여 압축 해제를 생략합니다.
임베딩 모델 로드 중 (Device: cuda)...
DB 로드 경로: ./chroma_db_bge_m3_data/chroma_db_bge_m3


In [ ]:
# from langchain_core.documents import Document

# # vector_db에서 문서 꺼냄
# raw_docs = vector_db.get(include=["metadatas", "documents"])

# # Document 객체 리스트로 변환
# all_documents = [
#     Document(page_content=doc, metadata=meta)
#     for doc, meta in zip(raw_docs["documents"], raw_docs["metadatas"])
# ]


In [ ]:
import json

def format_docs_for_evidence_wrapper(docs):
    formatted = []
    for i, doc in enumerate(docs):
        md = doc.metadata or {}

        metadata = {
            "author": md.get("author") or md.get("byline") or md.get("writer"),
            "category": md.get("category") or md.get("section") or md.get("topic"),
            "published_at": str(md.get("published_at") or md.get("published") or md.get("date") or ""),
            "source": md.get("source") or md.get("publisher") or md.get("site"),
            "title": (
                md.get("title")
                or md.get("original_title")
                or md.get("headline")
                or md.get("article_title")
                or md.get("document_title")
                or md.get("page_title")
                or "Untitled"  # 최후의 수단
            ),
            "url": (
                md.get("url")
                or md.get("original_url")
                or md.get("link")
                or md.get("source_url")
                or md.get("web_url")
            ),
        }

        content_block = f"DOCUMENT_ID: {i}\nCONTENT: {doc.page_content}\nMETADATA_JSON: {json.dumps(metadata, ensure_ascii=False)}"
        formatted.append(content_block)

    return "\n\n---\n\n".join(formatted)



In [ ]:

# 7.Retrieval 만들기 - 단계별 hop 설계


#1단계  : 1.넓게 먼저 유사한 문서 먼저 찾아서 정보 손실 없이 탐색기로 쓰기.
#         2.1과 관련해서 claim 쿼리와 비교해서 entity 추출
#2단계 : 1. 추출된 entity를 쿼리에 넣어서 다시 top-k로 추출 -> 이때는 근거 문서를 위한 용도이므로 k수를 줄여야 함.
#        2.다시 1과 관련해서 쿼리와 비교해서 entity 추출
#3단계 : 1,2와 동일 -> 이때도 안 되면 4단계, 일단 4단계가 최대라니까 이후는 최종 판결

#retriever = vector_db.as_retriever(search_kwargs={"k": 12})

# # 3. [핵심] 단계별 Multi-hop Retrieval 함수
# def multi_hop_retrieval_logic(question: str, vector_db, llm):
#     """
#     #1단계: 넓게 검색 -> Entity 추출
#     #2단계: Entity 포함 검색 -> 더 좁게 검색
#     #3단계: 동일
#     #4단계: 동일(최대 4hop)
#     """
#     accumulated_docs = []
#     current_query = question

#     # Entity 추출 체인
#     entity_prompt = ChatPromptTemplate.from_template(
#         "질문: {question}\n\n"
#         "위 질문을 검증하기 위해 추가로 검색해야 할 인물, 단체, 사건 등 핵심 단어(Entity)만 나열하세요. "
#         "(쉼표 구분, 없으면 None)"
#     )
#     entity_chain = entity_prompt | llm

#     # hop별 k (너 로직대로: 1단계 넓게, 뒤로 갈수록 좁게, 최대 4)
#     steps = [12, 6, 4, 3]

#     for i, k in enumerate(steps):
#         # ✅ MVP: 가장 단순한 similarity search
#         step_docs = vector_db.similarity_search(current_query, k=k)
#         accumulated_docs.extend(step_docs)

#         # 마지막 hop이면 entity 추출 안 하고 종료
#         if i == len(steps) - 1:
#             break

#         # entity 추출
#         new_entities_msg = entity_chain.invoke({"question": current_query})
#         new_entities = new_entities_msg.content.strip()

#         if "None" in new_entities or new_entities == "":
#             break

#         # 다음 hop 쿼리 확장
#         current_query = f"{question} {new_entities}"

#     # 중복 제거
#     unique_docs = {doc.page_content: doc for doc in accumulated_docs}.values()
#     return list(unique_docs)
##안되면 이거로##
# def multi_hop_retrieval_logic(question: str, vector_db, llm):
#     """
#     단계별 multi-hop retrieval:
#     - k를 점점 줄이며 재검색
#     - 각 단계에서 context 기반 entity 추출
#     - 누적 entity로 query 확장
#     """
#     accumulated_docs = []
#     all_entities = []
#     current_query = question

#     entity_prompt = ChatPromptTemplate.from_template(
#         """다음은 복잡한 검증 질문입니다:

# {question}

# [문서 요약 발췌]
# {context}

# 위 질문을 검증하기 위해 추가로 검색해야 할
# 인물, 단체, 사건 등의 핵심 단어(Entity)만 나열하세요.
# (쉼표 구분, 없으면 'None')"""
#     )
#     entity_chain = entity_prompt | llm

#     # narrowing 전략
#     steps = [12, 6, 3]

#     for i, k in enumerate(steps):
#         #  narrowing retrieval
#         step_docs = vector_db.similarity_search(current_query, k=k)
#         accumulated_docs.extend(step_docs)

#         if i < len(steps) - 1:
#             context_snippet = "\n".join(
#                 d.page_content[:200] for d in step_docs
#             )

#             new_entities = entity_chain.invoke({
#                 "question": question,
#                 "context": context_snippet
#             }).content.strip()

#             if new_entities and "None" not in new_entities:
#                 all_entities.append(new_entities)
#                 current_query = f"{question} {' '.join(all_entities)}"

#     # 중복 제거
#     unique_docs = {doc.page_content: doc for doc in accumulated_docs}.values()
#     return list(unique_docs)

def multi_hop_retrieval_logic(question: str, vector_db, llm):
    """
    #1단계: 넓게 검색 -> Entity 추출
    #2단계: Entity 포함 검색 -> 더 좁게 검색
    #3단계: 동일
    #4단계: 동일(최대 4hop)
    """
    accumulated_docs = []
    current_query = question

    # Entity 추출 체인
    entity_prompt = ChatPromptTemplate.from_template(
        "질문: {question}\n\n"
        "위 질문을 검증하기 위해 추가로 검색해야 할 인물, 단체, 사건 등 핵심 단어(Entity)만 나열하세요. "
        "(쉼표 구분, 없으면 None)"
    )
    entity_chain = entity_prompt | llm

    # hop별 k (너 로직대로: 1단계 넓게, 뒤로 갈수록 좁게, 최대 4)
    steps = [12, 6, 4, 3]

    for i, k in enumerate(steps):
        # ✅ MVP: 가장 단순한 similarity search
        step_docs = vector_db.similarity_search(current_query, k=k)
        accumulated_docs.extend(step_docs)

        # 마지막 hop이면 entity 추출 안 하고 종료
        if i == len(steps) - 1:
            break

        # entity 추출
        new_entities_msg = entity_chain.invoke({"question": current_query})
        new_entities = new_entities_msg.content.strip()

        if "None" in new_entities or new_entities == "":
            break

        # 다음 hop 쿼리 확장
        current_query = f"{question} {new_entities}"

    # 중복 제거
    unique_docs = {doc.page_content: doc for doc in accumulated_docs}.values()
    return list(unique_docs)

# def multi_hop_retrieval_logic(question: str, vector_db, llm):
#     """
#     Multi-hop retrieval logic:
#     1. Broad initial search
#     2. Check for relevant answer in docs
#     3. If not found, extract entities from context
#     4. Expand query and repeat up to 4 hops
#     """

#     accumulated_docs = []
#     current_query = question

#     # 단계별 검색 개수
#     steps = [12, 5, 4, 3]

#     # Entity 추출 체인
#     entity_prompt = ChatPromptTemplate.from_template(
#         "질문: {question}\n\n"
#         "현재까지 확인한 문서:\n{docs}\n\n"
#         "질문을 해결하기 위해 추가로 조사해야 할 핵심 단어(Entity)만 나열하세요. "
#         "(쉼표 구분, 없으면 None)"
#     )
#     entity_chain = entity_refinement_prompt | llm

#     # Query 확장 체인
#     query_expand_prompt = ChatPromptTemplate.from_template(
#         "원래 질문: {question}\n"
#         "다음 엔티티들을 반영하여 질문을 더 확장해 주세요: {entities}\n"
#         "확장된 질문:"
#     )
#     query_expand_chain = query_expand_prompt | llm

#     # Claim validation 체인
#     validation_prompt = ChatPromptTemplate.from_template(
#         "질문: {question}\n"
#         "문서 내용: {doc}\n\n"
#         "이 문서가 질문에 대한 답을 담고 있습니까? 예 / 아니오 / 불확실 중 하나로 답하세요."
#     )
#     validation_chain = validation_prompt | llm

#     for i, k in enumerate(steps):
#         # 🔍 Step 1: 검색
#         step_docs = vector_db.similarity_search(current_query, k=k)
#         accumulated_docs.extend(step_docs)

#         # ✅ Step 2: Claim validation
#         for doc in step_docs:
#             validation = validation_chain.invoke({
#                 "question": question,
#                 "doc": doc.page_content
#             }).content.strip()

#             if "예" in validation:
#                 # 답이 포함된 문서 발견 → 더 이상 hop 필요 없음
#                 return list({d.page_content: d for d in accumulated_docs}.values())

#         # ❌ Step 3: Entity 추출
#         step_docs_text = "\n\n".join([doc.page_content for doc in step_docs])
#         entity_result = entity_chain.invoke({
#             "question": question,
#             "docs": step_docs_text
#         }).content.strip()

#         if "None" in entity_result or not entity_result:
#             break  # 더 이상 진행할 수 없음

#         # ✨ Step 4: Query 확장
#         current_query = query_expand_chain.invoke({
#             "question": question,
#             "entities": entity_result
#         }).content.strip()

#     # 중복 제거 후 반환
#     unique_docs = {doc.page_content: doc for doc in accumulated_docs}.values()
#     return list(unique_docs)




In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field, RootModel

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

In [ ]:

from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class EvidenceItem(BaseModel):
    source: str = Field(default="unknown", description="출처(Source)")
    title: str = Field(default="", description="문서 제목(있으면)")
    url: str = Field(default="", description="출처 URL(있으면)")
    author: Optional[str] = Field(default=None, description="저자명(있으면)")  # 수정된 부분
    fact: str = Field(description="검증에 사용된 핵심 사실 문장 (Context에서 VERBATIM)")

class MultiHopStep(BaseModel):
    step: int = Field(description="추론 단계 번호")
    description: str = Field(description="해당 단계에서 수행한 분석 내용")

class FactCheckResult(BaseModel):
    # ✅ 프롬프트가 'answer'를 요구하므로 필드명을 answer로 맞춤
    answer: str = Field(description="최종 답변 (Yes/No/Insufficient information 또는 명사구)")

    # ✅ 프롬프트가 'evidence_list'를 요구하므로 필드명을 evidence_list로 맞춤
    evidence_list: List[EvidenceItem] = Field(default_factory=list, description="근거 리스트")

    # ✅ 프롬프트가 'multihop_reasoning'을 요구하므로 그대로 유지
    multihop_reasoning: List[MultiHopStep] = Field(default_factory=list, description="멀티홉 추론 과정")

    # (선택) 프롬프트 맨 위 분류를 결과로도 받고 싶으면 추가
    question_type: Optional[Literal["null_query","comparison_query","temporal_query","inference_query"]] = Field(
        default=None, description="질문 분류 결과(선택)"
    )


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

print("FactCheckResult defined?", "FactCheckResult" in globals())

FactCheckResult defined? True


In [ ]:
# ==============================
# 7. Entity Prompt 정의
# ==============================
from langchain_core.prompts import ChatPromptTemplate

entity_refinement_prompt = ChatPromptTemplate.from_template("""
[Entity Refinement]

Given the question and the following documents, list key entities (people, organizations, events, terms) that could lead to finding the answer.

Question: {question}

Context Documents:
{docs}

Only return the list of new entities or proper nouns, separated by commas. Do NOT repeat the question.

If no useful entities are found, return "None".
""")

In [ ]:


# OpenAI 공식 모델로 변경
llm = ChatOpenAI(
    model_name="gpt-4o-mini",  # 또는 성능과 비용 효율이 좋은 "gpt-4o-mini"
    temperature=0.0,

)

# ==============================
# 9. Prompt (System + Context + Question)
# ==============================
# 프롬프트를 생성합니다.
parser = PydanticOutputParser(pydantic_object=FactCheckResult)
format_instructions = parser.get_format_instructions()


prompt = ChatPromptTemplate.from_template("""

You are a multi-hop, evidence-based question answering AI.
Your task is to answer the question using the provided context and follow the reasoning steps.
[Question Classification]
 First, classify the question into one of these types:
 1) null_query: The context doesn't contain the answer.
 2) comparison_query: Requires comparing two or more entities/events.
 3) temporal_query: Involves time-based changes or sequences.
4) inference_query: Requires logical steps based on explicit facts.

[Answer Format Rules — MANDATORY]
Based on the question, the "answer" field MUST follow these rules:
A) For Yes/No Questions: - Output exactly "Yes", "No", or "Unknown". - No extra text or punctuation.
B) For Entity/Noun Questions (Who, What, Which, Name): - Output ONLY the noun or proper noun phrase. - If multiple, use a comma-separated list. - Do NOT write a full sentence.
C) If the answer cannot be found: - Output "Unknown".

 [Multi-hop Reasoning Procedure]
Step 1. Evidence Analysis
- Review the provided documents in the Context.
 - Even if the document doesn't have an exact word match, if it describes the fact or event in the question, use it as evidence.
 - If you find any relevant information, do NOT output 'Unknown'.

Step 2. Evidence Extraction
- Extract factual sentences VERBATIM from the selected documents.
- Do not paraphrase. Copy the exact text into 'evidence_list'.
- Ensure metadata (source, title, url, etc.) is correctly mapped for each evidence item.
Step 3. Final Answer Generation - Generate the "answer" strictly following the
[Answer Format Rules].
- The answer must be directly supported by the extracted evidence.

Step 4. Multi-hop Reasoning Explanation
- In 'multihop_reasoning', clearly explain the steps:
1) Which entities/facts were identified.
2) How the evidence sentences connect to each other.
3) How they lead to the final "Yes/No" or "Noun" answer.

[Output Requirements]
Your output MUST be valid JSON and strictly follow this schema:
{format_instructions}
Context: {context} Question: {question} Output:
""").partial(format_instructions=format_instructions)

In [ ]:

# 10. LCEL Chain (Retriever → Prompt → LLM → Output)

from functools import partial
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# multi-hop 함수에 retriever와 llm 고정
multi_hop_partial = partial(multi_hop_retrieval_logic, vector_db=vector_db, llm=llm)

# 체인 정의
chain = (
    {
        "context": RunnableLambda(multi_hop_partial) | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | parser
)

In [ ]:
# ==============================
# 11. 실행
# ==============================
#query = ("Do the TechCrunch article on software companies and the Hacker News article on The Epoch Times both report an increase in revenue related to payment and subscription models, respectively?")
#query = ("Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?")
#query = ("Considering the information from an article in The New York Times about the band Used To Be Young's latest tour and a review in Rolling Stone discussing the standout performance of a particular member during a recent concert, which member of Used To Be Young was highlighted for their exceptional solo during the tour's opening night and also plays the instrument that begins with the letter 'B'?")
#query=("Which company, according to articles from TechCrunch and The Verge, not only spent billions to maintain its default search engine status across various devices and platforms but was also considered by a major tech competitor as the only valid option for such services at the time of their deal, and is simultaneously facing a class action lawsuit for allegedly harming news publishers' revenues through its business practices?")
#query=("Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?")
#query=("Who is the individual associated with OpenAI, recognized for both his vision of AI agents and his generosity, and has made headlines in both Fortune and TechCrunch for his controversial departure from the company?")
#query=("Did the 'Fortune' report on Donald Trump's real estate valuations published on September 26, 2023, disagree with 'The Age' report regarding the allegation that Donald Trump increased the value of his penthouse apartment in the matter of inflating property values?")

response = chain.invoke(query)
print(response)

answer='No' evidence_list=[EvidenceItem(source='Hacker News', title='How the conspiracy-fueled Epoch Times went mainstream and made millions', url='https://news.ycombinator.com/item?id=123456', author=None, fact='The Epoch Times reported $76 million in subscription revenue in 2021, compared to nearly $7 million in 2019.'), EvidenceItem(source='The Verge', title='The new, ‘efficient’ Spotify has a very different approach to podcasting', url='https://www.theverge.com/2023/10/25/spotify-podcasting-ai', author=None, fact='Spotify looks to AI rather than original content for its podcasting future.')] multihop_reasoning=[MultiHopStep(step=1, description='Identified that The Epoch Times reported a significant increase in subscription revenue from $7 million in 2019 to $76 million in 2021.'), MultiHopStep(step=2, description='Identified that Spotify is focusing on AI for profitability rather than increasing revenue through original content.'), MultiHopStep(step=3, description='The evidence sho